In [0]:
 from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import DeltaTable

In [0]:
user_defined_Schema = StructType([

                                        StructField("storeId",IntegerType(),True),
                                        StructField("article_number",IntegerType()),
                                        StructField("qty",IntegerType(),True),
                                        StructField("reported_date",DateType(),True)


])
IO132_inbound_df = (spark.read.format("csv").
                    option("header",True)
                   .schema(user_defined_Schema).load('/mnt/inbound/I0132_inbound/'))
(IO132_inbound_df.write.mode("overwrite").
option('path',"/mnt/inventorycontainer/import_table/").saveAsTable("default.import_table")
)

# Outbound data frame 

outbound_table_instance = DeltaTable.forName(spark,"default.inventory_outbound_temp")
outbound_table_df = outbound_table_instance.toDF()


# Temp_stage_layer creation 

result_data_frame = (IO132_inbound_df.alias("t1").
 join(outbound_table_df.alias("t2")
                                   ,on=(col("t1.storeId")== col("t2.store_id")) & (col("t1.article_number") ==  col("t2.item_number")),
                                   how="left"
 
).select(
    col("t1.storeId").alias("store_id"),
    col("t1.article_number").alias("item_number"),
    col("t1.qty").alias("day1_qty"),
    when(col("t2.day1_qty").isNull(),0).otherwise(col("t2.day1_qty")).alias("day2_qty"),
    when(col("t2.day2_qty").isNull(),0).otherwise(col("t2.day2_qty")).alias("day3_qty"),
    when(col("t2.day3_qty").isNull(),0).otherwise(col("t2.day3_qty")).alias("day4_qty"),
    when(col("t2.day4_qty").isNull(),0).otherwise(col("t2.day4_qty")).alias("day5_qty"),
    when(col("t2.day5_qty").isNull(),0).otherwise(col("t2.day5_qty")).alias("day6_qty"),
    when(col("t2.day6_qty").isNull(),0).otherwise(col("t2.day6_qty")).alias("day7_qty")
         )
)



In [0]:

outbound_table_instance.alias("t1").merge(source= result_data_frame.alias("t2"),
                                          condition=(col("t1.store_id") == col("t2.store_Id")) & (col("t1.item_number") == col("t2.item_number"))
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()